# 🏡 San Antonio Automated Apartment & Townhome Hunter (SwRI Edition - v2)
### What's New in Version 2.0:
- **Hard Requirement Gatekeeper:** Automatically checks for **In-Unit Laundry** and **Dishwasher** (drops listings that explicitly only have shared/laundromat facilities or lack a dishwasher).
- **Garage & Bike Storage Bonus Module:** Parses property types (townhome/attached garage), parking fields, and descriptions for private garages, ground-floor access, and dedicated bike storage.
- **Costco Proximity Scorer:** Calculates drive time & road distance to all 4 San Antonio area Costcos (UTSA Blvd, Potranco/1604 West, Sonterra/1604 North, Selma).
- **Optional Extensible Modules Included:** Ready-to-toggle functions for Pet/Cat-Friendliness, Water Softener / Edwards Aquifer Hardness, Fiber Internet, and Flight-Path / Train-Horn Noise Buffers.

In [10]:
# @title Step 0: Install Required Packages
!pip install -q homeharvest osmnx geopandas folium shapely scipy requests google-genai

In [11]:
# @title Step 1: Configuration, Anchors & Weights (v2)
import os
import time
import json
import math
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
from shapely.geometry import Point
from scipy.spatial import cKDTree

CONFIG = {
    # Target Workplace: Southwest Research Institute (6220 Culebra Rd, San Antonio, TX)
    'swri_coords': (29.4485, -98.6105),

    # All 4 San Antonio Metro Costco Wholesale Locations
    'costco_locations': [
        {'name': 'Costco NW (5611 UTSA Blvd)', 'coords': (29.5762, -98.6001)},
        {'name': 'Costco West (191 W Loop 1604 S)', 'coords': (29.4316, -98.7115)},
        {'name': 'Costco North Central (1201 N Loop 1604 E)', 'coords': (29.6095, -98.4716)},
        {'name': 'Costco NE Selma (15330 IH-35 N)', 'coords': (29.5794, -98.3188)}
    ],

    # Baseline Property Filters
    'location': 'San Antonio, TX',
    'max_rent': 2200,          # Max monthly rent ($)
    'min_rent': 1200,          # Min monthly rent ($)
    'min_beds': 2,             # Minimum bedrooms
    'min_baths': 1.5,          # Minimum bathrooms

    # Hard Cutoffs
    'require_in_unit_laundry_and_dishwasher': True, # Enforce required appliances
    'strict_appliance_text_match': False, # Set True to drop listings that don't explicitly mention W/D in text
    'max_swri_commute_mins': 28.0,      # Auto-reject if rush-hour commute > 28 mins
    'max_grocery_dist_m': 2400.0,       # ~25 min walk or ~6 min bike ride (2.4 km)
    'max_park_trail_dist_m': 1600.0,    # ~18 min walk or ~4 min bike ride (1.6 km)

    # Updated v2 Scoring Weights (Sum = 1.0)
    'weights': {
        'commute_score': 0.22,
        'grocery_score': 0.16,
        'park_trail_score': 0.16,
        'pollution_highway_score': 0.12,
        'modern_interior_score': 0.16,
        'garage_bike_score': 0.10,      # NEW: Garage & E-Bike storage bonus
        'costco_score': 0.08            # NEW: Proximity to nearest Costco
    },

    # Optional API Keys
    'gemini_api_key': os.environ.get('GEMINI_API_KEY', '')
}

print('v2 Configuration loaded successfully!')

v2 Configuration loaded successfully!


In [12]:
# @title Step 2: Pull Geospatial Layers from OpenStreetMap (Groceries, Parks/Trails, Highways)
print('Fetching San Antonio geospatial layers from OpenStreetMap...')

def get_osm_coords(place, tags):
    gdf = ox.features_from_place(place, tags=tags)
    gdf_proj = gdf.to_crs(epsg=32614)
    centroids = gdf_proj.geometry.centroid.to_crs(epsg=4326)
    coords = np.array([(pt.y, pt.x) for pt in centroids])
    names = gdf['name'].fillna('Unnamed').tolist() if 'name' in gdf.columns else ['Unnamed']*len(coords)
    return coords, names, gdf_proj

grocery_coords, grocery_names, grocery_gdf = get_osm_coords(
    CONFIG['location'], {'shop': ['supermarket', 'grocery', 'greengrocer']}
)
park_coords, park_names, park_gdf = get_osm_coords(
    CONFIG['location'], {'leisure': ['park', 'nature_reserve'], 'highway': ['cycleway', 'path', 'footway']}
)
hwy_coords, hwy_names, hwy_gdf = get_osm_coords(
    CONFIG['location'], {'highway': ['motorway', 'trunk']}
)

grocery_xy = np.array([(geom.centroid.x, geom.centroid.y) for geom in grocery_gdf.geometry])
park_xy = np.array([(geom.centroid.x, geom.centroid.y) for geom in park_gdf.geometry])
hwy_xy = np.array([(geom.centroid.x, geom.centroid.y) for geom in hwy_gdf.geometry])

SPATIAL_TREES = {
    'grocery': (cKDTree(grocery_xy), grocery_names),
    'park': (cKDTree(park_xy), park_names),
    'highway': (cKDTree(hwy_xy), hwy_names)
}
print('Spatial KD-Trees built!')

Fetching San Antonio geospatial layers from OpenStreetMap...
Spatial KD-Trees built!


In [13]:
# @title Step 3: Automated Property Scraping (`HomeHarvest`)
from homeharvest import scrape_property

print('Scraping active rental listings in San Antonio...')
raw_listings = scrape_property(
    location=CONFIG['location'],
    listing_type='for_rent',
    past_days=30
)

df = raw_listings.copy()
df = df.dropna(subset=['latitude', 'longitude', 'list_price'])
df = df[
    (df['list_price'] >= CONFIG['min_rent']) &
    (df['list_price'] <= CONFIG['max_rent']) &
    (df['beds'].fillna(0) >= CONFIG['min_beds']) &
    (df['full_baths'].fillna(0) >= 1)
].copy()

gdf_listings = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs='EPSG:4326'
).to_crs(epsg=32614)

df['utm_x'] = gdf_listings.geometry.x
df['utm_y'] = gdf_listings.geometry.y
print(f'Listings matching budget & bed/bath specs: {len(df)}')

Scraping active rental listings in San Antonio...
Listings matching budget & bed/bath specs: 834


/tmp/ipykernel_6086/2445677117.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['beds'].fillna(0) >= CONFIG['min_beds']) &
/tmp/ipykernel_6086/2445677117.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['full_baths'].fillna(0) >= 1)


In [19]:
# @title Step 4: Modular Criteria Scoring Functions (Now with Laundry/Dishwasher Gate, Garage/Bike & Costco!)
CIRCUITY_FACTOR = 1.32
WALK_SPEED_M_MIN = 80.0
BIKE_SPEED_M_MIN = 250.0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def criterion_required_appliances_gatekeeper(row):
    """HARD REQUIREMENT: Checks for In-Unit Laundry (W/D or connections) & Dishwasher."""
    desc = str(row.get('description', '') or '').lower()
    year_built = row.get('year_built', np.nan)

    # Explicit disqualifiers for shared-only laundry
    shared_only_flags = ['shared laundry', 'laundromat on site', 'no washer', 'no laundry in unit', 'coin-operated laundry']
    if any(flag in desc for flag in shared_only_flags):
        return {'appliances_status': 'REJECT: Shared/No In-Unit Laundry', 'passes_appliance_gate': False}

    in_unit_terms = ['in-unit', 'in unit', 'washer/dryer', 'washer and dryer', 'w/d included', 'w/d connections', 'laundry room', 'utility room', 'full-size washer']
    has_in_unit = any(t in desc for t in in_unit_terms)
    has_dishwasher = 'dishwasher' in desc or ('stainless' in desc and 'appliances' in desc) or (pd.notna(year_built) and year_built >= 2005)

    if has_in_unit and has_dishwasher:
        status = 'Verified (In-Unit W/D + Dishwasher)'
        passes = True
    elif CONFIG['strict_appliance_text_match']:
        status = 'REJECT: Unconfirmed in Text'
        passes = False
    else:
        status = 'Likely (Verify W/D in Photos)'
        passes = True

    return {'appliances_status': status, 'passes_appliance_gate': passes}

def criterion_garage_and_bike_storage(row):
    """BONUS CRITERION: Scores private/attached garages, townhome ground access, & bike storage."""
    desc = str(row.get('description', '') or '').lower()
    prop_type = str(row.get('style', '') or row.get('type', '') or '').lower()
    parking_spaces = row.get('parking_garage', np.nan)

    score = 30.0 # Baseline (surface lot)
    features = []

    # Check structured garage column from HomeHarvest
    if pd.notna(parking_spaces) and float(parking_spaces) >= 1:
        score += 45.0
        features.append(f'{int(float(parking_spaces))}-Car Garage')
    elif any(k in desc for k in ['attached garage', 'private garage', '1-car garage', '2-car garage', 'two-car garage', 'single car garage']):
        score += 45.0
        features.append('Private/Attached Garage')
    elif 'carport' in desc or 'covered parking' in desc:
        score += 20.0
        features.append('Covered/Carport')

    # Check for bike storage / townhome ground-floor ease (critical for e-bikes)
    if any(k in desc for k in ['bike storage', 'bicycle storage', 'bike rack', 'bike repair', 'storage unit', 'storage closet']):
        score += 25.0
        features.append('Dedicated Bike/Gear Storage')
    if 'townhome' in prop_type or 'townhouse' in prop_type or 'townhome' in desc or 'first floor' in desc:
        score += 15.0
        features.append('Townhome/Ground Access')

    score = np.clip(score, 0.0, 100.0)
    return {
        'garage_bike_features': ', '.join(features) if features else 'Standard Surface Parking',
        'garage_bike_score': round(score, 1)
    }

def criterion_costco_proximity(row):
    """BONUS CRITERION: Distance and estimated drive time to nearest San Antonio Costco."""
    lat, lon = row['latitude'], row['longitude']
    best_name = ''
    min_road_km = 999.0

    for c in CONFIG['costco_locations']:
        clat, clon = c['coords']
        road_km = haversine_km(lat, lon, clat, clon) * 1.28
        if road_km < min_road_km:
            min_road_km = road_km
            best_name = c['name']

    est_drive_mins = (min_road_km / 48.0) * 60.0 + 2.0
    # Score: 100 if <= 7 mins drive, 0 if >= 25 mins drive
    score = np.clip(100 - ((est_drive_mins - 7.0) / (25.0 - 7.0)) * 100, 0, 100)
    return {
        'nearest_costco': best_name,
        'costco_drive_mins': round(est_drive_mins, 1),
        'costco_score': round(score, 1)
    }

def criterion_grocery_proximity(row):
    tree, names = SPATIAL_TREES['grocery']
    dist_euclidean, idx = tree.query([row['utm_x'], row['utm_y']])
    network_dist_m = dist_euclidean * CIRCUITY_FACTOR
    walk_mins = network_dist_m / WALK_SPEED_M_MIN
    bike_mins = network_dist_m / BIKE_SPEED_M_MIN
    score = np.clip(100 - ((network_dist_m - 400) / (2400 - 400)) * 100, 0, 100)
    return {
        'nearest_grocery': names[idx],
        'grocery_dist_m': round(network_dist_m),
        'grocery_walk_mins': round(walk_mins, 1),
        'grocery_bike_mins': round(bike_mins, 1),
        'grocery_score': round(score, 1)
    }

def criterion_park_trail_proximity(row):
    tree, names = SPATIAL_TREES['park']
    dist_euclidean, idx = tree.query([row['utm_x'], row['utm_y']])
    network_dist_m = dist_euclidean * 1.25
    walk_mins = network_dist_m / WALK_SPEED_M_MIN
    score = np.clip(100 - ((network_dist_m - 250) / (1600 - 250)) * 100, 0, 100)
    return {
        'nearest_park_trail': names[idx],
        'park_dist_m': round(network_dist_m),
        'park_walk_mins': round(walk_mins, 1),
        'park_trail_score': round(score, 1)
    }

def criterion_swri_rush_hour_commute(row):
    lat, lon = row['latitude'], row['longitude']
    swri_lat, swri_lon = CONFIG['swri_coords']
    road_km = haversine_km(lat, lon, swri_lat, swri_lon) * 1.30
    if lon < -98.68:
        avg_speed_kmh = 28.0
        corridor_note = 'West Inbound (Heavy Culebra/1604 Traffic)'
    elif lon > -98.58:
        avg_speed_kmh = 50.0
        corridor_note = 'Reverse Commute (Central/North)'
    else:
        avg_speed_kmh = 40.0
        corridor_note = 'Inner West / Loop 410 Local'
    est_commute_mins = (road_km / avg_speed_kmh) * 60.0 + 3.0
    score = np.clip(100 - ((est_commute_mins - 8.0) / (30.0 - 8.0)) * 100, 0, 100)
    return {
        'swri_road_km': round(road_km, 1),
        'swri_rush_hour_mins': round(est_commute_mins, 1),
        'commute_corridor': corridor_note,
        'commute_score': round(score, 1)
    }

def criterion_pollution_and_highway_buffer(row):
    tree, _ = SPATIAL_TREES['highway']
    hwy_dist_m, _ = tree.query([row['utm_x'], row['utm_y']])
    if hwy_dist_m < 100:
        score = 15.0
    elif hwy_dist_m < 250:
        score = 50.0 + ((hwy_dist_m - 100) / 150.0) * 35.0
    else:
        score = np.clip(85.0 + ((hwy_dist_m - 250) / 500.0) * 15.0, 85.0, 100.0)
    return {
        'highway_buffer_m': round(hwy_dist_m),
        'pollution_highway_score': round(score, 1)
    }

def criterion_modern_appliances_and_open_floorplan(row):
    desc = str(row.get('description', '') or '').lower()
    year_built = row.get('year_built', np.nan)
    score = 50.0
    matched_tags = []
    positive_keywords = {
        'open floor': 18, 'open-concept': 18, 'open concept': 18,
        'stainless steel': 15, 'stainless appliances': 15,
        'quartz': 12, 'granite': 10, 'kitchen island': 12, 'island kitchen': 12,
        'gas range': 12, 'gas stove': 12, 'modern': 8, 'renovated': 10
    }
    for kw, pts in positive_keywords.items():
        if kw in desc:
            score += pts
            matched_tags.append(kw)
    if pd.notna(year_built) and year_built >= 2018:
        score += 20
        matched_tags.append(f'built_{int(year_built)}')
    score = np.clip(score, 10.0, 100.0)
    return {
        'interior_tags': ', '.join(matched_tags) if matched_tags else 'Standard/Unlisted',
        'modern_interior_score': round(score, 1)
    }
def criterion_water_softener_and_filtration(row):
    """CRITERION: Detects built-in water softeners, softener loops, or reverse-osmosis taps
    to protect appliances from San Antonio hard limestone water (~18 gpg)."""
    desc = str(row.get('description', '') or '').lower()

    score = 40.0 # Baseline unlisted
    tags = []

    softener_keywords = {
        'water softener': 40, 'softener loop': 35, 'softened water': 35,
        'reverse osmosis': 25, 'ro system': 25, 'filtration system': 20,
        'water filtration': 20, 'tankless water heater': 15
    }

    for kw, pts in softener_keywords.items():
        if kw in desc:
            score += pts
            tags.append(kw)

    score = np.clip(score, 10.0, 100.0)
    return {
        'water_softener_tags': ', '.join(tags) if tags else 'Standard Hard Water (No Softener Listed)',
        'water_softener_score': round(float(score), 1)
    }

# Then add  to your CRITERIA_MODULES list!


CRITERIA_MODULES = [
    criterion_required_appliances_gatekeeper,
    criterion_garage_and_bike_storage,
    criterion_costco_proximity,
    criterion_grocery_proximity,
    criterion_park_trail_proximity,
    criterion_swri_rush_hour_commute,
    criterion_pollution_and_highway_buffer,
    criterion_modern_appliances_and_open_floorplan,
    criterion_water_softener_and_filtration
]
print(f'Registered {len(CRITERIA_MODULES)} modular scoring functions.')

Registered 9 modular scoring functions.


In [20]:
# @title Step 5: Execute v2 Pipeline, Filter by Hard Requirements & Export Map + CSV
print('Scoring all candidate listings across v2 criteria...')

results = []
for idx, row in df.iterrows():
    metrics = {}
    for module_fn in CRITERIA_MODULES:
        metrics.update(module_fn(row))

    w = CONFIG['weights']
    composite = (
        metrics['commute_score'] * w['commute_score'] +
        metrics['grocery_score'] * w['grocery_score'] +
        metrics['park_trail_score'] * w['park_trail_score'] +
        metrics['pollution_highway_score'] * w['pollution_highway_score'] +
        metrics['modern_interior_score'] * w['modern_interior_score'] +
        metrics['garage_bike_score'] * w['garage_bike_score'] +
        metrics['costco_score'] * w['costco_score']
    )
    metrics['composite_score'] = round(composite, 1)
    results.append(metrics)

scored_df = pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)

# Apply Hard Cutoffs (including In-Unit Laundry & Dishwasher Gate)
filtered_df = scored_df[
    (scored_df['passes_appliance_gate'] == True) &
    (scored_df['swri_rush_hour_mins'] <= CONFIG['max_swri_commute_mins']) &
    (scored_df['grocery_dist_m'] <= CONFIG['max_grocery_dist_m']) &
    (scored_df['park_dist_m'] <= CONFIG['max_park_trail_dist_m'])
].sort_values(by='composite_score', ascending=False).reset_index(drop=True)

print(f'Top surviving candidates after v2 cutoffs: {len(filtered_df)}')

display_cols = [
    'composite_score', 'formatted_address', 'list_price', 'beds', 'full_baths',
    'appliances_status', 'garage_bike_features', 'swri_rush_hour_mins',
    'nearest_costco', 'costco_drive_mins', 'nearest_grocery', 'grocery_walk_mins',
    'nearest_park_trail', 'park_walk_mins', 'highway_buffer_m', 'interior_tags', 'property_url'
]
existing_cols = [c for c in display_cols if c in filtered_df.columns]
filtered_df[existing_cols].to_csv('san_antonio_ranked_rentals_v2.csv', index=False)
print('Saved ranked spreadsheet to san_antonio_ranked_rentals_v2.csv')

# Generate Interactive HTML Map
m = folium.Map(location=[29.48, -98.58], zoom_start=11, tiles='CartoDB positron')

# Mark SwRI
folium.Marker(
    location=CONFIG['swri_coords'],
    popup='<b>Southwest Research Institute (SwRI)</b><br>6220 Culebra Rd',
    icon=folium.Icon(color='red', icon='briefcase', prefix='fa')
).add_to(m)

# Mark Costcos
for c in CONFIG['costco_locations']:
    folium.Marker(
        location=c['coords'],
        popup=f"<b>{c['name']}</b>",
        icon=folium.Icon(color='blue', icon='shopping-cart', prefix='fa')
    ).add_to(m)

# Plot Top 40 Ranked Properties
for _, r in filtered_df.head(40).iterrows():
    color = 'darkgreen' if r['composite_score'] >= 78 else ('green' if r['composite_score'] >= 65 else 'orange')
    popup_html = f"""
    <div style='width:280px; font-family:sans-serif;'>
        <h4 style='margin:0;'>Score: {r['composite_score']}/100 (${r['list_price']}/mo)</h4>
        <b>{r.get('formatted_address', 'Address')}</b><br>
        Beds/Baths: {r.get('beds')}bd / {r.get('full_baths')}ba<hr style='margin:4px 0;'>
        🧺 <b>W/D & Dishwasher:</b> {r['appliances_status']}<br>
        🚲 <b>Garage/Bike:</b> {r['garage_bike_features']}<br>
        🚗 <b>SwRI Rush Hour:</b> {r['swri_rush_hour_mins']} mins<br>
        📦 <b>Costco:</b> {r['costco_drive_mins']}m drive ({r['nearest_costco']})<br>
        🛒 <b>Grocery:</b> {r['grocery_walk_mins']}m walk / {r['grocery_bike_mins']}m bike ({r['nearest_grocery']})<br>
        🌲 <b>Park/Trail:</b> {r['park_walk_mins']}m walk ({r['nearest_park_trail']})<br>
        🌬️ <b>Highway Buffer:</b> {r['highway_buffer_m']}m<br>
        ✨ <b>Interior:</b> {r['interior_tags']}<br>
        <a href='{r.get('property_url', '#')}' target='_blank'>View Listing & Photos →</a>
    </div>
    """
    folium.Marker(
        location=[r['latitude'], r['longitude']],
        popup=folium.Popup(popup_html, max_width=320),
        icon=folium.Icon(color=color, icon='home', prefix='fa')
    ).add_to(m)

m.save('san_antonio_sweet_spots_map_v2.html')
print('Saved interactive map to san_antonio_sweet_spots_map_v2.html!')
filtered_df[existing_cols].head(15)

Scoring all candidate listings across v2 criteria...
Top surviving candidates after v2 cutoffs: 212
Saved ranked spreadsheet to san_antonio_ranked_rentals_v2.csv
Saved interactive map to san_antonio_sweet_spots_map_v2.html!


/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


,composite_score,formatted_address,list_price,beds,full_baths,appliances_status,garage_bike_features,swri_rush_hour_mins,nearest_costco,costco_drive_mins,nearest_grocery,grocery_walk_mins,nearest_park_trail,park_walk_mins,highway_buffer_m,interior_tags,property_url
0,81.3,"5142 Kelwood St Unit 102, San Antonio, TX, 78250",1600,3,2,Likely (Verify W/D in Photos),1-Car Garage,15.1,Costco West (191 W Loop 1604 S),14.0,H-E-B,6.8,Unnamed,1.0,3587,built_2025,https://www.realtor.com/rentals/details/5142-K...
1,78.2,"1022 Avocet, San Antonio, TX, 78245",1700,4,2,Likely (Verify W/D in Photos),2-Car Garage,13.1,Costco West (191 W Loop 1604 S),10.5,H-E-B Plus!,12.4,Unnamed,1.2,761,Standard/Unlisted,https://www.realtor.com/rentals/details/1022-A...
2,78.0,"4010 High Ridge Cir, San Antonio, TX, 78229",1950,3,2,Likely (Verify W/D in Photos),2-Car Garage,14.8,Costco NW (5611 UTSA Blvd),16.4,Minnano Japanese Grocery,5.7,Unnamed,0.9,1140,Standard/Unlisted,https://www.realtor.com/rentals/details/4010-H...
3,76.9,"4121 Copano Bay, San Antonio, TX, 78229",1700,3,2,Likely (Verify W/D in Photos),1-Car Garage,17.8,Costco NW (5611 UTSA Blvd),12.6,NOOR GROCERY,3.4,Unnamed,3.4,748,Standard/Unlisted,https://www.realtor.com/rentals/details/4121-C...
4,75.6,"5015 Shara Mist, San Antonio, TX, 78251",1495,3,2,Likely (Verify W/D in Photos),2-Car Garage,16.3,Costco West (191 W Loop 1604 S),13.2,Walmart Supercenter,9.6,Unnamed,2.4,2900,Standard/Unlisted,https://www.realtor.com/rentals/details/5015-S...
5,75.3,"8913 Fall River Dr, San Antonio, TX, 78250",1200,2,1,Likely (Verify W/D in Photos),1-Car Garage,17.0,Costco West (191 W Loop 1604 S),14.7,Walmart Supercenter,7.7,Unnamed,0.9,3565,Standard/Unlisted,https://www.realtor.com/rentals/details/8913-F...
6,75.3,"5011 Summit Wood Apt 2, San Antonio, TX, 78229",1495,3,2,Likely (Verify W/D in Photos),1-Car Garage,13.8,Costco NW (5611 UTSA Blvd),17.0,Walmart Supercenter,6.5,Unnamed,5.4,512,Standard/Unlisted,https://www.realtor.com/rentals/details/5011-S...
7,74.7,"7114 Faith Way Unit 102, San Antonio, TX, 78240",1695,3,2,Likely (Verify W/D in Photos),2-Car Garage,18.2,Costco NW (5611 UTSA Blvd),13.5,Walmart Supercenter,7.6,Unnamed,2.3,4669,Standard/Unlisted,https://www.realtor.com/rentals/details/7114-F...
8,74.6,"7110 Faith Way Unit 101, San Antonio, TX, 78240",1750,3,2,Likely (Verify W/D in Photos),2-Car Garage,18.2,Costco NW (5611 UTSA Blvd),13.5,Walmart Supercenter,7.9,Unnamed,2.5,4652,Standard/Unlisted,https://www.realtor.com/rentals/details/7110-F...
9,74.1,"5006 Summit Pass Apt 3, San Antonio, TX, 78229",1450,3,2,Likely (Verify W/D in Photos),1-Car Garage,13.8,Costco NW (5611 UTSA Blvd),17.0,Walmart Supercenter,7.1,Unnamed,6.1,483,Standard/Unlisted,https://www.realtor.com/rentals/details/5006-S...
